#### Install libraries
- !pip install langchain
- !pip install langchain-community
- !pip install langchain-huggingface
- !pip install langchain-text-splitters
- !pip install pypdf
- !pip install faiss-cpu

In [ ]:
# Load Libraries
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import os
from itertools import chain

In [ ]:
user_prompt = "What is supervised learning?"

In [ ]:
# Helper function for printing docs
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

### Create a Prompt Template

#### Input Keys:
- context = retrieved documents
- chat_history = memory of previous interaction(s)
- query = user's request (includes requirements)

In [ ]:
template = """<s>[INST] You are an AI assistant tasked with answering questions. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: 
{context}

Question:
{query}
Answer: [/INST]"""

### LLM Model Set-Up

Parameters:

* max_tokens (int, default: 200 ) – The maximum number of tokens to generate.
* temp (float, default: 0.7 ) – The model temperature. Larger values increase creativity but decrease factuality.
* top_k (int, default: 40 ) – Randomly sample from the top_k most likely tokens at each generation step. Set this to 1 for greedy decoding.
* top_p (float, default: 0.4 ) – Randomly sample at each generation step from the top most likely tokens whose probabilities add up to top_p.
* min_p (float, default: 0.0 ) – Randomly sample at each generation step from the top most likely tokens whose probabilities are at least min_p.
* repeat_penalty (float, default: 1.18 ) – Penalize the model for repetition. Higher values result in less repetition.
* repeat_last_n (int, default: 64 ) – How far in the models generation history to apply the repeat penalty.
* n_batch (int, default: 8 ) – Number of prompt tokens processed in parallel. Larger values decrease latency but increase resource requirements.

In [ ]:
def get_llm(path, temp=0.1, max_tokens=1024, top_p=0.4, top_k=40):
    # Load the model from a local path
    llm = HuggingFacePipeline.from_model_id(
        model_id=path,
        task="text-generation",
        model_kwargs={
                  "do_sample": True, 
                  "temperature": temp, 
                  "max_length": max_tokens,
                  "top_p": top_p, 
                  "top_k": top_k
        }
    )
        
    return(llm)

In [ ]:
llm = get_llm("models--mistralai--Mistral-7B-Instruct-v0.2/snapshots/41b61a33a2483885c981aa79e0df6b32407ed873")

### Load Documents for RAG Retrieval

In [ ]:
def load_documents(documents = ['bioconf_iscku2024_00133.pdf', 'Deep Kernel Principal Component Analysis for multi-level feature learning.pdf']):
    # Load uploaded documents
    data_documents = []
    
    for doc in documents:
        if doc.endswith("pdf"):
            data = PyPDFLoader(doc).load()
        elif doc.endswith("docx"):
            data = Docx2txtLoader(doc).load()
        elif doc.endswith("txt"):
            data = TextLoader(doc).load()
        data_documents.append(data)

    text_documents = list(chain.from_iterable(data_documents))
    return(text_documents)

In [ ]:
text_documents = load_documents()

In [ ]:
text_documents[:3]

### Split Documents with Chunking Strategy

In [ ]:
def split_documents(text_documents, chunk_size=500, chunk_overlap=64):
    # Split documents
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    split_text = text_splitter.split_documents(text_documents)
    return(split_text)

In [ ]:
documents_split = split_documents(text_documents, 500, 64)

In [ ]:
documents_split[:3]

### Create a Vector Store 

From:
* Loaded and Split Documents 
* Embeddings Model

In [ ]:
def create_vectorstore(documents_split, embeddings_path="nomic-ai/nomic-embed-text-v1.5"):
    
    # load embeddings
    embeddings = HuggingFaceEmbeddings(model_name=embeddings_path, 
                                       model_kwargs={"trust_remote_code": True},
                                       encode_kwargs={'normalize_embeddings': True})
    
    # Create vectorstore from embeddings
    index = FAISS.from_documents(documents_split, embeddings)
    
    return(embeddings, index)

In [ ]:
embeddings, vectorstore = create_vectorstore(documents_split)

### Create Retriever from the Vector Store

In [ ]:
def create_retriever(vectorstore, search_type, search_params: dict):
    """
    Create retriver using a vector store
        - Maximum marginal relevance: "mmr", search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_param':0.5}
        - Similarity score threshold: "similarity_score_threshold", search_kwargs={"score_threshold": 0.5}
        - Specifying top k: "similarity", search_kwargs={'k': 6}
    """
    retriever = vectorstore.as_retriever(search_type=search_type, search_kwargs=search_params)
    return(retriever)

In [ ]:
retriever = create_retriever(vectorstore, "similarity", {"k": 3})

### Return the Context

Returns the retrieved documents (context) from the user_prompt

In [ ]:
def get_context(retriever, prompt):
    # Select retriever type      
    compressed_docs = retriever.invoke(prompt)       
    pretty_print_docs(compressed_docs)
    
    context = ""
    doc_text = []
    source = []
    for doc in compressed_docs:            
        context = context + doc.page_content + "\n"
        doc_text.append(doc.page_content)
        source.append(doc.metadata)
        
    return(context, source, doc_text)

In [ ]:
context, source, doc_text = get_context(retriever, prompt=user_prompt)

### Invoke LLM

In [ ]:
template.format(context=context, query=user_prompt)

In [ ]:
response = llm.invoke(template.format(context=context, query=user_prompt))

In [ ]:
response.split('Answer: [/INST] ')[1]